# CerviGrade: retrain the CNNs on CRIC with fewer Negative cells per epoch
Same recipe as the earlier retrain (SIPaKMeD starting checkpoints, early stopping on validation macro F1, three seeds, same slide-level split, same 1,723 test cells), with one change: **each epoch uses only `NEGATIVE_PER_EPOCH` randomly chosen Negative cells** (default 1,000, about the size of the largest abnormal classes). The draw is new every epoch, so no Negative cell is thrown away for good. Class weights start switched off (`CLASS_WEIGHT_POWER = 0`), so the effect of the cap shows on its own.

**Inputs to add:** the CRIC dataset (`cervigrade-cric-dataset`), the Stage 1 `Training notebook` output, and your earlier retrain results Dataset (only used for the comparison in the last cell). GPU on. No Internet needed.

**To run it:** run cells 1 to 5 (EfficientNetB0 only, about 15 to 20 minutes). Skip cells 6 and 8 unless you want ConvNeXt and RegNet too. Then run cells 9, 10 and the comparison in cell 11. The test set is never used to choose anything.

## 1. Settings
The only cell you edit. Set `QUICK_TEST = True` first to check everything runs (a few minutes), then set it back to `False`.

In [1]:
# ----- Retrain settings (the only cell you edit) -----
SEEDS = [1, 2, 3]            # one full training run per seed, for each model
RT_EPOCHS = 30               # upper limit; early stopping usually ends sooner
RT_PATIENCE = 5              # stop after this many epochs without a better validation macro F1
RT_LR = 1e-4
RT_DROPOUT = 0.3
CLASS_WEIGHT_POWER = 0.0     # 0 = off, 1 = full balancing. The earlier runs used 0.5. This run starts at 0, so the effect of the Negative cap shows on its own.
RT_IMG = 224
RT_BATCH = 32
RT_WORKERS = 0              # 0 = load images in the main process. Safer after TensorFlow has started (extra worker processes can freeze RegNet)
QUICK_TEST = False           # True = one seed and one epoch per model, to check that everything runs
NEGATIVE_PER_EPOCH = 1000    # Each epoch uses this many randomly chosen Negative cells, a new draw every epoch, so no cell is lost for good. None = every Negative cell, every epoch (the earlier runs).
SAVE_MODELS = True          # True = also save the three chosen models as .keras / .pt files (slower, large files). The main notebook only needs the probabilities.

if QUICK_TEST:
    SEEDS, RT_EPOCHS, RT_PATIENCE = [1], 1, 1
    print("QUICK TEST: one seed and one epoch per model. The scores mean nothing.")
RT_ID = f"w{CLASS_WEIGHT_POWER}" + (f"_n{NEGATIVE_PER_EPOCH}" if NEGATIVE_PER_EPOCH else "") + f"_e{RT_EPOCHS}_p{RT_PATIENCE}"
RT_DIR = f"/kaggle/working/runs_{RT_ID}"
print("Finished runs are saved in", RT_DIR)

Finished runs are saved in /kaggle/working/runs_w0.0_n1000_e30_p5


## 2. Find the data and build the split
Same split as the main notebook: by source slide, seed 42, 70 / 15 / 15. The test count should be 1,723.

In [2]:
import os, re, random
import pandas as pd

def find_containing(root, predicate, max_matches=5):
    matches = []
    for dirpath, dirnames, filenames in os.walk(root):
        if predicate(dirpath, dirnames, filenames):
            matches.append(dirpath)
            if len(matches) >= max_matches:
                break
    return matches

INPUT_ROOT = "/kaggle/input"
cric_matches = find_containing(INPUT_ROOT, lambda d, dn, fn: os.path.basename(d) == "CRIC_cropped")
print("CRIC_cropped found at:", cric_matches)
CRIC_CROPPED_DIR = cric_matches[0]

ckpt_files = {"efficientnetb0": "efficientnetb0_sipakmed_best.keras",
              "convnext": "convnext_sipakmed_best.keras",
              "regnet": "regnet_sipakmed_best.pt"}
checkpoints = {}
for key, fname in ckpt_files.items():
    hits = find_containing(INPUT_ROOT, lambda d, dn, fn, f=fname: f in fn, max_matches=1)
    checkpoints[key] = os.path.join(hits[0], fname) if hits else None
    print(f"  {key}: {'found' if hits else 'MISSING'} -> {checkpoints[key]}")
assert all(checkpoints.values()), "Add the Stage 1 'training-notebook' output as an input."

RANDOM_SEED = 42
def build_cric_manifest():
    rows = []
    for cls in sorted(os.listdir(CRIC_CROPPED_DIR)):
        cls_dir = os.path.join(CRIC_CROPPED_DIR, cls)
        if not os.path.isdir(cls_dir):
            continue
        for fname in os.listdir(cls_dir):
            slide_id = fname.split("_")[0]
            rows.append({"filepath": os.path.join(cls_dir, fname), "label": cls,
                         "dataset": "CRIC", "group": f"CRIC_slide_{slide_id}"})
    return pd.DataFrame(rows)

def split_by_group_global(manifest_df, seed=RANDOM_SEED, train_frac=0.7, val_frac=0.15):
    rng = random.Random(seed)
    groups = sorted(manifest_df["group"].unique())
    rng.shuffle(groups)
    n = len(groups)
    n_train = round(n * train_frac)
    n_val = round(n * val_frac)
    group_split = {}
    for g in groups[:n_train]:
        group_split[g] = "train"
    for g in groups[n_train:n_train + n_val]:
        group_split[g] = "val"
    for g in groups[n_train + n_val:]:
        group_split[g] = "test"
    return manifest_df["group"].map(group_split)

cric_manifest = build_cric_manifest()
cric_manifest["split"] = split_by_group_global(cric_manifest)

CRIC_cropped found at: ['/kaggle/input/datasets/wagnermushayija/cervigrade-cric-dataset/CRIC_cropped']
  efficientnetb0: found -> /kaggle/input/notebooks/wagnermushayija/training-notebook/efficientnetb0_sipakmed_best.keras
  convnext: found -> /kaggle/input/notebooks/wagnermushayija/training-notebook/convnext_sipakmed_best.keras
  regnet: found -> /kaggle/input/notebooks/wagnermushayija/training-notebook/regnet_sipakmed_best.pt


## 3. Class weights, scoring and helpers
Rare grades get a larger weight in the loss, computed from the training split only. This cell also holds the small helpers the training cells use.

In [3]:
import os, time, json, copy, gc, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, accuracy_score, f1_score

missing = [n for n in ("CRIC_CROPPED_DIR", "checkpoints", "cric_manifest") if n not in globals()]
assert not missing, f"Run the cells above first. Missing: {missing}"
os.makedirs(RT_DIR, exist_ok=True)

CKPT = {"EfficientNetB0": checkpoints["efficientnetb0"], "ConvNeXt": checkpoints["convnext"], "RegNet": checkpoints["regnet"]}
for arch, path in CKPT.items():
    assert os.path.isfile(path), f"Missing Stage 1 checkpoint for {arch}: {path}"

cric_classes = sorted(cric_manifest["label"].unique())
cric_class_to_idx = {c: i for i, c in enumerate(cric_classes)}
SHORT_NAMES = [c.replace("Negative for intraepithelial lesion", "Negative") for c in cric_classes]
RT_K = len(cric_classes)
cric_train_df = cric_manifest[cric_manifest["split"] == "train"]
cric_val_df = cric_manifest[cric_manifest["split"] == "val"]
cric_test_df = cric_manifest[cric_manifest["split"] == "test"]
rt_y_train = cric_train_df["label"].map(cric_class_to_idx).values
rt_y_val = cric_val_df["label"].map(cric_class_to_idx).values
y_true = cric_test_df["label"].map(cric_class_to_idx).values
rt_test_keys = (cric_test_df["label"] + "/" + cric_test_df["filepath"].map(os.path.basename)).values
print(f"CRIC  train {len(cric_train_df)}  val {len(cric_val_df)}  test {len(cric_test_df)}   (test should be 1,723)")

# Rare grades get a larger weight in the loss. The weights come from the training split only.
NEG_IDX = next(i for i, c in enumerate(cric_classes) if c.startswith("Negative"))
rt_counts = np.bincount(rt_y_train, minlength=RT_K)
if NEGATIVE_PER_EPOCH:
    rt_counts[NEG_IDX] = min(rt_counts[NEG_IDX], NEGATIVE_PER_EPOCH)       # cells the model really sees in one epoch
rt_w = (rt_counts.sum() / (RT_K * rt_counts)) ** CLASS_WEIGHT_POWER
rt_w = rt_w * rt_counts.sum() / (rt_w * rt_counts).sum()          # keep the average weight per image at 1
RT_CLASS_WEIGHTS = {i: float(rt_w[i]) for i in range(RT_K)}
print(pd.DataFrame({"train cells per epoch": rt_counts, "weight": np.round(rt_w, 2)}, index=SHORT_NAMES))

def rt_test_metrics(y_true, y_prob):
    y_pred = np.argmax(y_prob, axis=1)
    rep = classification_report(y_true, y_pred, target_names=SHORT_NAMES, output_dict=True, zero_division=0)
    out = {"accuracy": round(accuracy_score(y_true, y_pred), 3),
           "weighted_f1": round(rep["weighted avg"]["f1-score"], 3),
           "macro_f1": round(rep["macro avg"]["f1-score"], 3)}
    for c in SHORT_NAMES:
        out[f"recall_{c}"] = round(rep[c]["recall"], 3)
    return out

# ----- Bring back earlier results (only matters in a fresh session) -----
# A fresh session starts with an empty /kaggle/working. If you attached a Kaggle Dataset that holds the downloaded runs_* folder,
# its files are copied back here, so finished seeds are skipped instead of trained again.
import shutil
if not os.path.exists(os.path.join(RT_DIR, "runs.json")):
    found = [d for d, _, fns in os.walk(INPUT_ROOT) if "runs.json" in fns]
    # Only restore runs made with the same settings: a folder with the same name, or (with no Negative cap) a flat download of the earlier runs.
    candidates = [d for d in found if os.path.basename(d) == os.path.basename(RT_DIR)]
    if not candidates and not NEGATIVE_PER_EPOCH:
        candidates = [d for d in found if not os.path.basename(d).startswith("runs_")]
    if candidates:
        for fn in os.listdir(candidates[0]):
            if os.path.isfile(os.path.join(candidates[0], fn)):
                shutil.copy2(os.path.join(candidates[0], fn), os.path.join(RT_DIR, fn))
        print("Restored earlier runs from", candidates[0])
    else:
        print("No earlier runs with these settings found in the inputs: everything will train from scratch.")

RT_RUNS_FILE = os.path.join(RT_DIR, "runs.json")
rt_runs = json.load(open(RT_RUNS_FILE)) if os.path.exists(RT_RUNS_FILE) else {}
def rt_probs_path(arch, seed): return os.path.join(RT_DIR, f"probs_{arch}_seed{seed}.npz")
def rt_weights_path(arch, seed): return os.path.join(RT_DIR, f"weights_{arch}_seed{seed}" + (".pt" if arch == "RegNet" else ".weights.h5"))
def rt_run_done(arch, seed): return f"{arch}_seed{seed}" in rt_runs and os.path.exists(rt_probs_path(arch, seed))
print(f"{len(rt_runs)} finished run(s) found on disk.")

# ----- Helpers used by the three training cells -----
from IPython.display import display
RT_METRIC_COLS = ["accuracy", "weighted_f1", "macro_f1"] + [f"recall_{c}" for c in SHORT_NAMES]

def rt_mean_std(series):
    m, s = series.mean(), series.std(ddof=1)
    return f"{m:.3f} ± {s:.3f}" if len(series) > 1 else f"{m:.3f} (1 run)"

def rt_train_all(arch):
    """Train one model once per seed. Finished runs are skipped, so it is safe to run again."""
    durations = []
    for n, seed in enumerate(SEEDS, 1):
        key = f"{arch}_seed{seed}"
        if rt_run_done(arch, seed):
            print(f"[{n}/{len(SEEDS)}] {key}: already finished, skipping")
            continue
        t0 = time.time()
        info, probs = rt_run_regnet(seed) if arch == "RegNet" else rt_run_keras(arch, seed)
        minutes = (time.time() - t0) / 60
        durations.append(minutes)
        np.savez(rt_probs_path(arch, seed), y_prob=probs, y_true=y_true, test_keys=rt_test_keys)
        rt_runs[key] = {"architecture": arch, "seed": seed, "minutes": round(minutes, 1), **info, **rt_test_metrics(y_true, probs)}
        json.dump(rt_runs, open(RT_RUNS_FILE, "w"), indent=2)
        r = rt_runs[key]
        left = sum(1 for s in SEEDS[n:] if not rt_run_done(arch, s))
        print(f"[{n}/{len(SEEDS)}] {key}: {minutes:.1f} min | best epoch {r['best_epoch']} of {r['epochs_run']} | "
              f"val macro F1 {r['val_macro_f1']} | test acc {r['accuracy']} weighted F1 {r['weighted_f1']} "
              f"macro F1 {r['macro_f1']} SCC recall {r['recall_SCC']} | about {left * np.mean(durations):.0f} min left")
    rt_show(arch)

def rt_show(arch):
    sub = pd.DataFrame([r for r in rt_runs.values() if r["architecture"] == arch]).sort_values("seed")
    if QUICK_TEST:
        print("QUICK TEST: these scores come from one epoch and mean nothing.")
    print(f"\n{arch}: every seed")
    display(sub[["seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS].reset_index(drop=True))
    print(f"{arch}: average ± standard deviation across seeds")
    display(pd.DataFrame({c: [rt_mean_std(sub[c])] for c in RT_METRIC_COLS}, index=[f"{arch} ({len(sub)} runs)"]))

CRIC  train 8004  val 1807  test 1723   (test should be 1,723)
          train cells per epoch  weight
ASC-H                       651     1.0
ASC-US                      425     1.0
HSIL                        906     1.0
LSIL                       1079     1.0
Negative                   1000     1.0
SCC                         104     1.0
No earlier runs with these settings found in the inputs: everything will train from scratch.
0 finished run(s) found on disk.


## 4. Keras training code (EfficientNetB0 and ConvNeXt)
Definitions only. Nothing trains here.

In [4]:
import tensorflow as tf
try:
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.set_logical_device_configuration(gpu, [tf.config.LogicalDeviceConfiguration(memory_limit=10240)])
except Exception as e:
    print("GPU memory limit left as it was:", e)    # harmless; TensorFlow was already started by an earlier cell
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input as rt_effnet_pre
from tensorflow.keras.applications.convnext import preprocess_input as rt_convnext_pre
RT_PREPROCESS = {"EfficientNetB0": rt_effnet_pre, "ConvNeXt": rt_convnext_pre}

def rt_make_tf_dataset(df, preprocess_fn, shuffle_seed=None, augment=False):
    filepaths = df["filepath"].values
    labels = df["label"].map(cric_class_to_idx).values
    def load_and_preprocess(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [RT_IMG, RT_IMG])
        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_flip_up_down(image)
        return preprocess_fn(image), label
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle_seed is not None:
        ds = ds.shuffle(buffer_size=len(df), seed=shuffle_seed)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RT_BATCH).prefetch(tf.data.AUTOTUNE)

def rt_build_keras(arch):
    loaded = tf.keras.models.load_model(CKPT[arch])
    base = next(l for l in loaded.layers if isinstance(l, tf.keras.Model))
    base.trainable = True
    inputs = layers.Input(shape=(RT_IMG, RT_IMG, 3))
    x = base(inputs)
    x = layers.Dropout(RT_DROPOUT)(x)
    outputs = layers.Dense(RT_K, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=RT_LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def rt_make_train_dataset(preprocess_fn, seed):
    """Training data. With NEGATIVE_PER_EPOCH set, every epoch uses a fresh random subset of the Negative cells."""
    if not NEGATIVE_PER_EPOCH:
        return rt_make_tf_dataset(cric_train_df, preprocess_fn, shuffle_seed=seed, augment=True)
    filepaths = cric_train_df["filepath"].values
    labels = cric_train_df["label"].map(cric_class_to_idx).values
    is_neg = labels == NEG_IDX
    def load_and_augment(filepath, label):
        image = tf.io.read_file(filepath)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [RT_IMG, RT_IMG])
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_flip_up_down(image)
        return preprocess_fn(image), label
    negatives = tf.data.Dataset.from_tensor_slices((filepaths[is_neg], labels[is_neg]))
    negatives = negatives.shuffle(int(is_neg.sum()), seed=seed, reshuffle_each_iteration=True).take(NEGATIVE_PER_EPOCH)
    others = tf.data.Dataset.from_tensor_slices((filepaths[~is_neg], labels[~is_neg]))
    ds = negatives.concatenate(others).shuffle(len(cric_train_df), seed=seed + 1, reshuffle_each_iteration=True)
    ds = ds.map(load_and_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(RT_BATCH).prefetch(tf.data.AUTOTUNE)

class MacroF1Stopping(tf.keras.callbacks.Callback):
    def __init__(self, val_ds, y_val, patience):
        super().__init__()
        self.val_ds, self.y_val, self.patience = val_ds, y_val, patience
        self.best, self.best_epoch, self.best_weights, self.wait = -1.0, 0, None, 0
    def on_epoch_end(self, epoch, logs=None):
        probs = self.model.predict(self.val_ds, verbose=0)
        f1 = f1_score(self.y_val, probs.argmax(1), average="macro", zero_division=0)
        logs["val_macro_f1"] = f1
        if f1 > self.best + 1e-6:
            self.best, self.best_epoch, self.wait = f1, epoch + 1, 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)

class RtEpochBar(tf.keras.callbacks.Callback):
    def __init__(self, max_epochs, title):
        super().__init__()
        self.max_epochs, self.title = max_epochs, title
    def on_train_begin(self, logs=None):
        self.outer = tqdm(total=self.max_epochs, desc=self.title, unit="epoch")
    def on_epoch_begin(self, epoch, logs=None):
        self.inner = tqdm(total=self.params.get("steps"), desc=f"  epoch {epoch + 1}", unit="batch", leave=False)
    def on_train_batch_end(self, batch, logs=None):
        self.inner.update(1)
    def on_epoch_end(self, epoch, logs=None):
        self.inner.close()
        self.outer.update(1)
        self.outer.set_postfix(loss=f"{logs['loss']:.3f}", acc=f"{logs['accuracy']:.3f}", val_macro_f1=f"{logs['val_macro_f1']:.3f}")
    def on_train_end(self, logs=None):
        self.outer.close()

def rt_run_keras(arch, seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    pre = RT_PREPROCESS[arch]
    train_ds = rt_make_train_dataset(pre, seed)
    val_ds = rt_make_tf_dataset(cric_val_df, pre)
    test_ds = rt_make_tf_dataset(cric_test_df, pre)
    model = rt_build_keras(arch)
    stopper = MacroF1Stopping(val_ds, rt_y_val, RT_PATIENCE)
    history = model.fit(train_ds, epochs=RT_EPOCHS, class_weight=RT_CLASS_WEIGHTS, verbose=0,
                        callbacks=[stopper, RtEpochBar(RT_EPOCHS, f"{arch} seed {seed}")])
    probs = model.predict(test_ds, verbose=0)
    model.save_weights(rt_weights_path(arch, seed))
    info = {"best_epoch": stopper.best_epoch, "epochs_run": len(history.history["loss"]), "val_macro_f1": round(stopper.best, 3)}
    del model, train_ds, val_ds, test_ds
    tf.keras.backend.clear_session(); gc.collect()
    return info, probs

## 5. Train EfficientNetB0
One run per seed, then its table. Finished runs are saved, so running the cell again skips them.

In [5]:
rt_train_all("EfficientNetB0")

I0000 00:00:1792445738.131201      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 10240 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792445738.136258      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 10240 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


EfficientNetB0 seed 1:   0%|          | 0/30 [00:00<?, ?epoch/s]

  epoch 1:   0%|          | 0/131 [00:00<?, ?batch/s]

2026-10-19 21:36:34.299906: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 21:36:34.442483: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 21:36:34.792967: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 21:36:34.936448: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 21:36:35.077214: E external/local_xla/xla/stream_

  epoch 2:   0%|          | 0/131 [00:00<?, ?batch/s]

...

  epoch 12:   0%|          | 0/131 [00:00<?, ?batch/s]

[3/3] EfficientNetB0_seed3: 5.8 min | best epoch 7 of 12 | val macro F1 0.561 | test acc 0.83 weighted F1 0.828 macro F1 0.587 SCC recall 0.265 | about 0 min left

EfficientNetB0: every seed


,seed,best_epoch,epochs_run,val_macro_f1,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
0,1,13,18,0.573,0.810,0.811,0.569,0.510,0.250,0.785,0.550,0.938,0.588
1,2,10,15,0.573,0.823,0.820,0.581,0.490,0.241,0.833,0.574,0.944,0.471
2,3,7,12,0.561,0.830,0.828,0.587,0.571,0.361,0.868,0.550,0.934,0.265


EfficientNetB0: average ± standard deviation across seeds


,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
EfficientNetB0 (3 runs),0.821 ± 0.010,0.820 ± 0.009,0.579 ± 0.009,0.524 ± 0.042,0.284 ± 0.067,0.829 ± 0.042,0.558 ± 0.014,0.939 ± 0.005,0.441 ± 0.164


## 6. Train ConvNeXt-Tiny

In [6]:
rt_train_all("ConvNeXt")

ConvNeXt seed 1:   0%|          | 0/30 [00:00<?, ?epoch/s]

  epoch 1:   0%|          | 0/131 [00:00<?, ?batch/s]

2026-10-19 22:05:24.783578: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 22:05:24.919153: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 22:07:21.385539: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 22:07:21.520225: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-10-19 22:07:52.203025: E external/local_xla/xla/stream_

  epoch 2:   0%|          | 0/131 [00:00<?, ?batch/s]

  epoch 3:   0%|          | 0/131 [00:00<?, ?batch/s]

...

  epoch 13:   0%|          | 0/131 [00:00<?, ?batch/s]

[3/3] ConvNeXt_seed3: 22.4 min | best epoch 8 of 13 | val macro F1 0.588 | test acc 0.832 weighted F1 0.829 macro F1 0.596 SCC recall 0.412 | about 0 min left

ConvNeXt: every seed


,seed,best_epoch,epochs_run,val_macro_f1,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
0,1,15,20,0.605,0.820,0.814,0.583,0.347,0.398,0.775,0.395,0.972,0.588
1,2,6,11,0.558,0.796,0.800,0.586,0.571,0.389,0.708,0.512,0.932,0.618
2,3,8,13,0.588,0.832,0.829,0.596,0.571,0.315,0.871,0.504,0.943,0.412


ConvNeXt: average ± standard deviation across seeds


,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
ConvNeXt (3 runs),0.816 ± 0.018,0.814 ± 0.015,0.588 ± 0.007,0.496 ± 0.129,0.367 ± 0.046,0.785 ± 0.082,0.470 ± 0.065,0.949 ± 0.021,0.539 ± 0.111


## 7. RegNetY-400MF training code (PyTorch)
Definitions only. Nothing trains here.

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import regnet_y_400mf
from PIL import Image

rt_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch device:", rt_device)

_mean, _std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
rt_train_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.RandomHorizontalFlip(),
                                  transforms.RandomVerticalFlip(), transforms.ToTensor(), transforms.Normalize(_mean, _std)])
rt_eval_tf = transforms.Compose([transforms.Resize((RT_IMG, RT_IMG)), transforms.ToTensor(), transforms.Normalize(_mean, _std)])

class RtCricDataset(Dataset):
    def __init__(self, df, transform):
        self.filepaths = df["filepath"].values
        self.labels = df["label"].map(cric_class_to_idx).values
        self.transform = transform
    def __len__(self):
        return len(self.filepaths)
    def __getitem__(self, idx):
        return self.transform(Image.open(self.filepaths[idx]).convert("RGB")), int(self.labels[idx])

def rt_build_regnet():
    model = regnet_y_400mf(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, 5)          # the 5-class SIPaKMeD head, so the saved weights fit
    model.load_state_dict(torch.load(CKPT["RegNet"], map_location="cpu"))
    model.fc = torch.nn.Sequential(torch.nn.Dropout(RT_DROPOUT), torch.nn.Linear(model.fc.in_features, RT_K))
    for p in model.parameters():
        p.requires_grad = True
    return model.to(rt_device)

def rt_torch_probs(model, loader, desc):
    model.eval()
    out = []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc=desc, unit="batch", leave=False):
            out.append(torch.softmax(model(images.to(rt_device)), dim=1).cpu().numpy())
    return np.concatenate(out)

class RtNegativeCapSampler(torch.utils.data.Sampler):
    """Every epoch: all non-Negative cells plus a fresh random subset of the Negative cells."""
    def __init__(self, labels, neg_idx, cap, seed):
        labels = np.asarray(labels)
        self.neg = np.where(labels == neg_idx)[0]
        self.rest = np.where(labels != neg_idx)[0]
        self.cap = min(cap, len(self.neg))
        self.rng = np.random.default_rng(seed)
    def __len__(self):
        return len(self.rest) + self.cap
    def __iter__(self):
        idx = np.concatenate([self.rest, self.rng.choice(self.neg, self.cap, replace=False)])
        self.rng.shuffle(idx)
        return iter(idx.tolist())

def rt_run_regnet(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    g = torch.Generator(); g.manual_seed(seed)
    train_set = RtCricDataset(cric_train_df, rt_train_tf)
    if NEGATIVE_PER_EPOCH:
        train_loader = DataLoader(train_set, batch_size=RT_BATCH, sampler=RtNegativeCapSampler(train_set.labels, NEG_IDX, NEGATIVE_PER_EPOCH, seed), num_workers=RT_WORKERS)
    else:
        train_loader = DataLoader(train_set, batch_size=RT_BATCH, shuffle=True, num_workers=RT_WORKERS, generator=g)
    val_loader = DataLoader(RtCricDataset(cric_val_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    test_loader = DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, shuffle=False, num_workers=RT_WORKERS)
    model = rt_build_regnet()
    optimizer = torch.optim.Adam(model.parameters(), lr=RT_LR)
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([RT_CLASS_WEIGHTS[i] for i in range(RT_K)], dtype=torch.float32, device=rt_device))
    best, best_epoch, best_state, wait, epochs_run = -1.0, 0, None, 0, 0
    bar = tqdm(range(RT_EPOCHS), desc=f"RegNet seed {seed}", unit="epoch")
    for epoch in bar:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"  epoch {epoch + 1}", unit="batch", leave=False):
            images, labels = images.to(rt_device), labels.to(rt_device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
        epochs_run += 1
        f1 = f1_score(rt_y_val, rt_torch_probs(model, val_loader, "  validating").argmax(1), average="macro", zero_division=0)
        bar.set_postfix(loss=f"{total_loss / total:.3f}", acc=f"{correct / total:.3f}", val_macro_f1=f"{f1:.3f}")
        if f1 > best + 1e-6:
            best, best_epoch, wait = f1, epoch + 1, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            wait += 1
            if wait >= RT_PATIENCE:
                break
    model.load_state_dict(best_state)
    probs = rt_torch_probs(model, test_loader, "testing")
    torch.save(model.state_dict(), rt_weights_path("RegNet", seed))
    info = {"best_epoch": best_epoch, "epochs_run": epochs_run, "val_macro_f1": round(best, 3)}
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return info, probs

PyTorch device: cuda


## 8. Train RegNetY-400MF

In [8]:
rt_train_all("RegNet")

RegNet seed 1:   0%|          | 0/30 [00:00<?, ?epoch/s]

  epoch 1:   0%|          | 0/131 [00:00<?, ?batch/s]

  validating:   0%|          | 0/57 [00:00<?, ?batch/s]

  epoch 2:   0%|          | 0/131 [00:00<?, ?batch/s]

  validating:   0%|          | 0/57 [00:00<?, ?batch/s]

...

testing:   0%|          | 0/54 [00:00<?, ?batch/s]

[3/3] RegNet_seed3: 14.9 min | best epoch 12 of 17 | val macro F1 0.542 | test acc 0.812 weighted F1 0.818 macro F1 0.579 SCC recall 0.529 | about 0 min left

RegNet: every seed


,seed,best_epoch,epochs_run,val_macro_f1,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
0,1,18,23,0.568,0.844,0.846,0.649,0.551,0.444,0.878,0.628,0.927,0.559
1,2,8,13,0.524,0.786,0.797,0.559,0.510,0.361,0.751,0.643,0.892,0.412
2,3,12,17,0.542,0.812,0.818,0.579,0.388,0.389,0.782,0.605,0.929,0.529


RegNet: average ± standard deviation across seeds


,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
RegNet (3 runs),0.814 ± 0.029,0.820 ± 0.025,0.596 ± 0.047,0.483 ± 0.085,0.398 ± 0.042,0.804 ± 0.066,0.625 ± 0.019,0.916 ± 0.021,0.500 ± 0.078


## 9. All three side by side
Every run, then the average ± standard deviation per model. Differences smaller than the spread between seeds are noise.

In [9]:
# ----- All three models side by side -----
rt_order = {a: i for i, a in enumerate(["EfficientNetB0", "ConvNeXt", "RegNet"])}
rt_df = pd.DataFrame(rt_runs.values())
assert len(rt_df), "No finished runs yet. Run the three training cells above first."
rt_df = rt_df.assign(_o=rt_df["architecture"].map(rt_order)).sort_values(["_o", "seed"]).drop(columns="_o").reset_index(drop=True)
missing_models = [a for a in rt_order if a not in set(rt_df["architecture"])]
if missing_models:
    print("Not trained yet:", missing_models)
if QUICK_TEST:
    print("QUICK TEST: the scores below come from one epoch and mean nothing.")
print("Every run")
display(rt_df[["architecture", "seed", "best_epoch", "epochs_run", "val_macro_f1"] + RT_METRIC_COLS])

rt_summary = rt_df.groupby("architecture")[RT_METRIC_COLS].agg(rt_mean_std)
rt_summary = rt_summary.loc[[a for a in rt_order if a in rt_summary.index]]
rt_summary.insert(0, "runs", rt_df.groupby("architecture").size())
print("Average ± standard deviation across seeds")
display(rt_summary)
rt_summary.to_csv("/kaggle/working/retrain_summary.csv")
rt_df.to_csv("/kaggle/working/retrain_all_runs.csv", index=False)

Every run


,architecture,seed,best_epoch,epochs_run,val_macro_f1,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
0,EfficientNetB0,1,13,18,0.573,0.810,0.811,0.569,0.510,0.250,0.785,0.550,0.938,0.588
1,EfficientNetB0,2,10,15,0.573,0.823,0.820,0.581,0.490,0.241,0.833,0.574,0.944,0.471
2,EfficientNetB0,3,7,12,0.561,0.830,0.828,0.587,0.571,0.361,0.868,0.550,0.934,0.265
3,ConvNeXt,1,15,20,0.605,0.820,0.814,0.583,0.347,0.398,0.775,0.395,0.972,0.588
4,ConvNeXt,2,6,11,0.558,0.796,0.800,0.586,0.571,0.389,0.708,0.512,0.932,0.618
5,ConvNeXt,3,8,13,0.588,0.832,0.829,0.596,0.571,0.315,0.871,0.504,0.943,0.412
6,RegNet,1,18,23,0.568,0.844,0.846,0.649,0.551,0.444,0.878,0.628,0.927,0.559
7,RegNet,2,8,13,0.524,0.786,0.797,0.559,0.510,0.361,0.751,0.643,0.892,0.412
8,RegNet,3,12,17,0.542,0.812,0.818,0.579,0.388,0.389,0.782,0.605,0.929,0.529


Average ± standard deviation across seeds


,runs,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
architecture,,,,,,,,,,
EfficientNetB0,3,0.821 ± 0.010,0.820 ± 0.009,0.579 ± 0.009,0.524 ± 0.042,0.284 ± 0.067,0.829 ± 0.042,0.558 ± 0.014,0.939 ± 0.005,0.441 ± 0.164
ConvNeXt,3,0.816 ± 0.018,0.814 ± 0.015,0.588 ± 0.007,0.496 ± 0.129,0.367 ± 0.046,0.785 ± 0.082,0.470 ± 0.065,0.949 ± 0.021,0.539 ± 0.111
RegNet,3,0.814 ± 0.029,0.820 ± 0.025,0.596 ± 0.047,0.483 ± 0.085,0.398 ± 0.042,0.804 ± 0.066,0.625 ± 0.019,0.916 ± 0.021,0.500 ± 0.078


## 10. Pick the seed for each model and save the results
The seed with the best **validation** macro F1 is kept for each model. Its stored test probabilities are written to `/kaggle/working`, which is all the main notebook reads. With `SAVE_MODELS = True` the models themselves are also rebuilt, saved, reloaded and checked.

In [10]:
# ----- Pick one seed per model by validation score and save the results -----
rt_best_seed = {}
for arch in rt_order:
    sub = rt_df[rt_df["architecture"] == arch]
    if len(sub):
        rt_best_seed[arch] = int(sub.sort_values(["val_macro_f1", "seed"], ascending=[False, True]).iloc[0]["seed"])
print("Chosen seed per model (best validation macro F1):", rt_best_seed)

rt_bundle = {"y_true": y_true, "test_keys": rt_test_keys}
for arch, seed in rt_best_seed.items():
    rt_bundle[arch] = np.load(rt_probs_path(arch, seed))["y_prob"]
np.savez("/kaggle/working/stage2_retrain_probs.npz", **rt_bundle)
json.dump({"best_seed": rt_best_seed, "settings_id": RT_ID, "class_weight_power": CLASS_WEIGHT_POWER, "negative_per_epoch": NEGATIVE_PER_EPOCH},
          open("/kaggle/working/retrain_selection.json", "w"), indent=2)
print("Saved stage2_retrain_probs.npz and retrain_selection.json")

if SAVE_MODELS:
    # Rebuild each chosen model, save it, reload it and check it against the stored test probabilities.
    for arch, seed in rt_best_seed.items():
        if arch == "RegNet":
            m = rt_build_regnet()
            m.load_state_dict(torch.load(rt_weights_path(arch, seed), map_location=rt_device))
            check = rt_torch_probs(m, DataLoader(RtCricDataset(cric_test_df, rt_eval_tf), batch_size=RT_BATCH, num_workers=RT_WORKERS), "re-checking")
            torch.save(m.state_dict(), "/kaggle/working/regnet_cric_retrained.pt")
            del m
        else:
            tf.keras.backend.clear_session()
            m = rt_build_keras(arch)
            m.load_weights(rt_weights_path(arch, seed))
            check = m.predict(rt_make_tf_dataset(cric_test_df, RT_PREPROCESS[arch]), verbose=0)
            m.save(f"/kaggle/working/{arch.lower()}_cric_retrained.keras")
            del m
        gap = np.abs(check - rt_bundle[arch]).max()
        print(f"{arch}: saved the seed {seed} model. Reloaded model matches the stored test probabilities (largest gap {gap:.4f}).")
        if gap > 0.02:
            print("  WARNING: the reloaded model does not match. Do not use this saved file.")
else:
    print("SAVE_MODELS is False: the model files were not saved. The weights stay in the runs folder.")

Chosen seed per model (best validation macro F1): {'EfficientNetB0': 1, 'ConvNeXt': 1, 'RegNet': 1}
Saved stage2_retrain_probs.npz and retrain_selection.json


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 428 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


EfficientNetB0: saved the seed 1 model. Reloaded model matches the stored test probabilities (largest gap 0.0000).


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 366 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


ConvNeXt: saved the seed 1 model. Reloaded model matches the stored test probabilities (largest gap 0.0000).


re-checking:   0%|          | 0/54 [00:00<?, ?batch/s]

RegNet: saved the seed 1 model. Reloaded model matches the stored test probabilities (largest gap 0.0000).


## 11. Compare with the earlier run
Same test cells, same metrics. Differences smaller than the spread between seeds are noise.

In [11]:
# ----- Compare with the earlier run (class weights 0.5, every Negative cell every epoch) -----
import glob
base_files = glob.glob("/kaggle/input/**/retrain_all_runs.csv", recursive=True)
if not base_files:
    print("Attach the earlier retrain results Dataset to see the comparison.")
else:
    base = pd.read_csv(base_files[0])
    rows = {}
    for arch in [a for a in rt_order if a in set(rt_df["architecture"])]:
        earlier = base[base["architecture"] == arch]
        this_run = rt_df[rt_df["architecture"] == arch]
        if len(earlier):
            rows[f"{arch}: earlier (weights 0.5, all Negative cells)"] = {c: rt_mean_std(earlier[c]) for c in RT_METRIC_COLS}
        rows[f"{arch}: this run (Negative cap {NEGATIVE_PER_EPOCH}, weights {CLASS_WEIGHT_POWER})"] = {c: rt_mean_std(this_run[c]) for c in RT_METRIC_COLS}
    comparison = pd.DataFrame(rows).T
    display(comparison)
    comparison.to_csv("/kaggle/working/retrain_comparison.csv")
    print("Read it by recall: Negative recall shows the false alarms on healthy cells, SCC recall shows the cancer-grade catch rate, macro F1 shows the overall balance. Accuracy is expected to drop, because the test set is still 59% Negative.")

,accuracy,weighted_f1,macro_f1,recall_ASC-H,recall_ASC-US,recall_HSIL,recall_LSIL,recall_Negative,recall_SCC
"EfficientNetB0: earlier (weights 0.5, all Negative cells)",0.824 ± 0.004,0.826 ± 0.003,0.606 ± 0.004,0.503 ± 0.071,0.321 ± 0.043,0.836 ± 0.020,0.527 ± 0.041,0.933 ± 0.011,0.696 ± 0.095
"EfficientNetB0: this run (Negative cap 1000, weights 0.0)",0.821 ± 0.010,0.820 ± 0.009,0.579 ± 0.009,0.524 ± 0.042,0.284 ± 0.067,0.829 ± 0.042,0.558 ± 0.014,0.939 ± 0.005,0.441 ± 0.164
"ConvNeXt: earlier (weights 0.5, all Negative cells)",0.826 ± 0.019,0.831 ± 0.016,0.613 ± 0.026,0.544 ± 0.077,0.380 ± 0.047,0.823 ± 0.055,0.623 ± 0.045,0.928 ± 0.013,0.519 ± 0.136
"ConvNeXt: this run (Negative cap 1000, weights 0.0)",0.816 ± 0.018,0.814 ± 0.015,0.588 ± 0.007,0.496 ± 0.129,0.367 ± 0.046,0.785 ± 0.082,0.470 ± 0.065,0.949 ± 0.021,0.539 ± 0.111
"RegNet: earlier (weights 0.5, all Negative cells)",0.805 ± 0.009,0.809 ± 0.005,0.576 ± 0.008,0.524 ± 0.042,0.364 ± 0.051,0.766 ± 0.017,0.569 ± 0.052,0.926 ± 0.018,0.480 ± 0.237
"RegNet: this run (Negative cap 1000, weights 0.0)",0.814 ± 0.029,0.820 ± 0.025,0.596 ± 0.047,0.483 ± 0.085,0.398 ± 0.042,0.804 ± 0.066,0.625 ± 0.019,0.916 ± 0.021,0.500 ± 0.078


Read it by recall: Negative recall shows the false alarms on healthy cells, SCC recall shows the cancer-grade catch rate, macro F1 shows the overall balance. Accuracy is expected to drop, because the test set is still 59% Negative.
